# mcp

> The MCP frontend: `Client` as tools on stdio

In [ ]:
#| default_exp mcp

The frontend Claude Code launches per conversation: `mk_server` closes the tools over a `Client`, and `main` (the `clikernel-mcp` console script) serves it on stdio via mcpmini. There is no instructions machinery and nothing eager — usage is taught by skill text, the server answers `initialize` instantly, and nothing happens until the model calls `connect`. Tool descriptions carry v1's hard-won wording.


In [ ]:
#| export
import asyncio
from fastcore.utils import *
from mcpmini.core import MCPServer, serve_stdio
from aidialog.dialog import Message
from aidialog.hist import output_parts, merge_media
from aidialog.msg_parts import PartType, data_url
from clikernel.core import Client, render_outs
from clikernel import __version__


In [ ]:
from fastcore.test import *
import base64
from io import BytesIO
from PIL import Image as PImg
import os, socket, tempfile
from clikernel.core import STATE_LOST
from jupygate.core import create_app, serve


Image outputs ride the shared aidialog machinery: `output_parts` runs the gate/resize/tag pipeline and `merge_media` composes the result with the rendered text — a plain string when there are no images (byte-identical to the text-only path), a `Part` list otherwise. The one piece that must live here is the wire edge: MCP's content-block vocabulary.

In [ ]:
#| export
def part2block(p):
    "An MCP content block for aidialog `Part` `p`"
    if p.type != PartType.input_image: return dict(type='text', text=p.text)
    mime,b64 = data_url(p.text)
    return dict(type='image', data=b64, mimeType=mime)

In [ ]:
#| export
def mk_server(c:Client):
    "An `MCPServer` whose tools close over `c`: connect, execute, and the lifecycle verbs"
    async def connect(
        host:str='', # Gateway: empty for the default local jupygate, a `gateways.toml` name, or a URL
        kernel:str='', # Kernel id (or unique prefix) to attach to; empty creates a fresh kernel
    )->str:
        "Connect to a kernel. With `kernel`: attach to that existing kernel exactly as it is (nothing is run) - this is how a later conversation returns to live state, and how to reach a kernel someone else created. Without: create a fresh kernel, run the user's startup.py in it, and install their inspectors; the reply includes the new kernel's id (reusable in a later `connect`) and the startup output. Kernels persist until explicitly stopped: disconnecting, switching, and conversation end never kill anything."
        return await c.connect(host, kernel)

    async def execute(
        code:str, # Python/IPython code to run
    ):
        "Run `code` in the current kernel, keeping state across calls (imports, variables, monkeypatches, cached objects). Requires a `connect` first. If the reply says the kernel died, `connect` again. Image outputs (plots etc.) come back as image blocks, resized to a token-friendly size, each preceded by its `<media id=...>` tag."
        r = await c.execute_outs(code)
        if isinstance(r, str): return r
        res = merge_media(render_outs(r), output_parts(Message(msg_type='code', output=r)))
        return res if isinstance(res, str) else dict(content=[part2block(p) for p in res], isError=False)

    async def list_kernels(
        host:str='', # Gateway to list; empty for the current one (or the default if not connected)
    )->str:
        "One line per kernel on the gateway: id, state, connection count, and which is current. Use it to find a kernel to reattach to, or to spot forgotten kernels worth stopping."
        return await c.list_kernels(host)

    async def stop_kernel(
        kernel:str='', # Kernel id (or unique prefix); empty stops the current kernel
    )->str:
        "Stop a kernel for good: its process ends and its state is gone. This is the only way any kernel is ever stopped - do it when the user is done with it, and leave kernels running that the user wants to return to."
        return await c.stop(kernel)

    async def restart()->str:
        "Kill the current kernel's process and start a fresh one under the same id: `sys.modules` genuinely reset, all session state discarded. Use for a clean slate, after rebuilding a native extension, or after a reload left stale classes behind (symptoms: `isinstance` mysteriously failing, a class missing a method you know it has). Also works when `execute` is stuck. After restarting, redo any imports/setup the task still needs."
        return await c.restart()

    async def interrupt()->str:
        "Interrupt the code the current kernel is running (SIGINT, i.e. KeyboardInterrupt): the in-flight `execute` returns with a KeyboardInterrupt traceback, and session state survives. Prefer this over `restart` when a call is merely taking too long. Only meaningful while an `execute` is running."
        return await c.interrupt()

    return MCPServer('clikernel', [connect, execute, list_kernels, stop_kernel, restart, interrupt], version=__version__)


Directly at the dispatch level, against a live in-thread jupygate: the tools carry real schemas from their docments, `connect` returns the id and banner, and state persists across `execute` calls:


In [ ]:
def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

gport = free_port()
gserver = serve(create_app(), port=gport, in_thread=True)
os.environ['CLIKERNEL_HOST'] = f'http://127.0.0.1:{gport}'
cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('base = 42; print("ready")')

c = Client(cfgd)
srv = mk_server(c)
def call(name, **kw): return dict(jsonrpc='2.0', id=1, method='tools/call', params=dict(name=name, arguments=kw))
r = await srv.dispatch(dict(jsonrpc='2.0', id=0, method='tools/list', params={}))
[t['name'] for t in r['result']['tools']]


['connect', 'execute', 'list_kernels', 'stop_kernel', 'restart', 'interrupt']

In [ ]:
def txt(r): return r['result']['content'][0]['text']
r = await srv.dispatch(call('connect'))
kid = re.search(r'created kernel (\w+)', txt(r)).group(1)
assert 'ready' in txt(r)
r = await srv.dispatch(call('execute', code='base * 2'))
test_eq(txt(r), '84')
r = await srv.dispatch(call('restart'))
test_eq(txt(r), STATE_LOST)
r = await srv.dispatch(call('list_kernels'))
assert kid in txt(r) and 'current' in txt(r)
r = await srv.dispatch(call('stop_kernel'))
assert 'stopped' in txt(r)


Images through the same dispatch path: a fresh kernel renders an oversized PIL image, and the reply is a content list — the `<media>` tag text block, the image block (resized under aidialog's `im_max` pixel budget), and the rendered text last, with no base64 leaking into any text block. An image that fails to decode folds into plain text as a `<media-unavailable>` note instead of failing the call, and text-only executes stay plain strings, byte-identical to before.

In [ ]:
r = await srv.dispatch(call('connect'))
r = await srv.dispatch(call('execute', code='from PIL import Image\nImage.new("RGB", (1000, 800), "red")'))
blocks = r['result']['content']
test_eq([b['type'] for b in blocks], ['text', 'image', 'text'])
assert '<media' in blocks[0]['text']
test_eq(blocks[1]['mimeType'], 'image/jpeg')   # PIL's repr publishes png *and* jpeg; `IMG_MIMES` order picks jpeg - the one-image-per-output dedup at work
im = PImg.open(BytesIO(base64.b64decode(blocks[1]['data'])))
from aidialog.hist import im_max
assert im.width*im.height <= im_max
assert 'base64' not in blocks[2]['text']

In [ ]:
r = await srv.dispatch(call('execute', code='from IPython.display import display\ndisplay({"image/png": "not-base64"}, raw=True)\n"after"'))
assert '<media-unavailable' in txt(r) and 'after' in txt(r)
r = await srv.dispatch(call('execute', code='40+2'))
test_eq(txt(r), '42')
r = await srv.dispatch(call('stop_kernel'))
assert 'stopped' in txt(r)

`main` is the `clikernel-mcp` entry point, and it is intentionally nothing: make a `Client`, serve the tools on stdio, close the connections (never a kernel) on the way out. No eager work — `initialize` answers instantly, so one-shot hosts (like the Agent SDK) see the tools immediately.


In [ ]:
#| export
def main():
    "The `clikernel-mcp` console script: the tools on stdio, state is one `Client`"
    async def _main():
        c = Client()
        try: await serve_stdio(mk_server(c))
        finally: await c.aclose()
    asyncio.run(_main())


End to end the way Claude Code runs it: the installed `clikernel-mcp` script as a subprocess, driven by mcpmini's own client. `initialize` answers before any kernel exists; `connect` creates one against the live gateway; a second conversation (a second subprocess) attaches to the same kernel and finds the state.


In [ ]:
import shutil
from mcpmini.core import MCPClient


In [ ]:
cmd = shutil.which('clikernel-mcp')
assert cmd, 'clikernel-mcp script not installed: run `uv sync`'
tdir = Path(tempfile.mkdtemp())   # empty config dir: no startup, no inspectors
env = os.environ | {'CLIKERNEL_HOST': f'http://127.0.0.1:{gport}', 'XDG_CONFIG_HOME': str(tdir)}

async with MCPClient.stdio([cmd], env=env) as m1:
    r = await m1.tools.connect()
    kid = re.search(r'created kernel (\w+)', r).group(1)
    await m1.tools.execute(code='x = 6*7')
    renv = await m1.tools.execute(code='import os; os.environ["XDG_CONFIG_HOME"]')
    assert str(tdir) in renv   # the kernel got the conversation's env, not the gateway's

async with MCPClient.stdio([cmd], env=env) as m2:   # the next conversation
    r = await m2.tools.connect(kernel=kid[:8])
    assert 'existing' in r
    res = await m2.tools.execute(code='x')
    await m2.tools.stop_kernel()
res


'42'

## A live client

The proof that matters for deployment: Claude Code's real MCP client, driven headless by the Agent SDK, calling the new tool surface — `connect` first, then `execute` — against a live gateway. `#| eval: false`: it spends tokens; rerun manually when the tools or Claude Code move.


In [ ]:
#| eval: false
import logging, random
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage

In [ ]:
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_dir = Path(tempfile.mkdtemp())
lenv = {k:str(v) for k,v in (os.environ | {'CLIKERNEL_HOST': f'http://127.0.0.1:{gport}', 'XDG_CONFIG_HOME': str(live_dir)}).items()}
opts = ClaudeAgentOptions(mcp_servers=dict(ck=dict(type='stdio', command=cmd, env=lenv)), cwd=str(live_dir),
    allowed_tools=['mcp__ck__connect','mcp__ck__execute'], max_turns=6)
prompt = 'Using the ck MCP tools: first connect, then compute 17*19 in the kernel. Reply with just the number.'
msgs = [m async for m in query(prompt=prompt, options=opts)]
tus = {b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if isinstance(b, ToolUseBlock)}
res = first(m.result for m in msgs if isinstance(m, ResultMessage))
assert {'mcp__ck__connect','mcp__ck__execute'} <= tus
assert '323' in res
res


'323'

And the image path with real Claude: the test host picks a color the model is never told, the kernel renders it as a plain image, and the model answers from the image block alone — proof the block survives dispatch, stdio framing, and the host's own client, and lands in the model's vision context.

In [ ]:
#| eval: false
color = random.choice(['red', 'green', 'blue', 'yellow', 'purple', 'orange'])
(live_dir/'color.txt').write_text(color)
code = "from PIL import Image\nImage.new('RGB', (200,200), open('color.txt').read().strip())"
prompt = ('Using the ck MCP tools: first connect, then execute exactly this code (do not run anything else, and do not read color.txt any other way):\n\n'
    f'{code}\n\nThe execute result includes an image. Reply with just the color of that image.')
msgs2 = [m async for m in query(prompt=prompt, options=opts)]
res2 = first(m.result for m in msgs2 if isinstance(m, ResultMessage))
assert color in res2.lower()
color, res2

('purple', 'Purple.')

In [ ]:
#|hide
gserver.should_exit = True

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()